# SYNAPSE CHANNEL — getting started

The local-first coordination bus for fleets of AI agents. This notebook runs the
shipped self-contained demo — no repository, no configuration — so you can watch
the coordination model happen, then shows the Python client you would call from
your own code.

Companion prose: [Tutorial: watch two agents coordinate](https://github.com/anulum/synapse-channel/blob/main/docs/tutorial.md).

## 1. Install

The core install has a single runtime dependency (`websockets`); everything else
is the standard library.

In [ ]:
%pip install --quiet synapse-channel

## 2. Watch coordination happen

`synapse demo` starts its own local hub, seats two agents, and drives a short
coordination flow. Watch for three lines: a **mutation denied** (a claim gates
the change), an **atomic handoff**, and a **verified receipt**. It ends with
`success: coordination demo completed`.

In [ ]:
!synapse demo

## 3. What you just saw

1. **A claim gates the mutation.** An edit to `src/shared.py` without a claim on
   it was refused *before* it landed — a file-scope claim is the one thing that
   authorises a change, so two agents never edit the same file at once.
2. **Handoff moves authority atomically.** The held scope, status, and checkpoint
   moved to the other agent in one step — no window where the file is unowned.
3. **The result is evidence, not a promise.** The receipt records that real
   checks ran and hashes the changed files, so coordination is auditable.

No database, cloud, or consensus protocol was involved — one local hub owned the
state and both agents connected to it.

## 4. Drive it from Python

To coordinate from your own code, start a durable hub in a terminal first:

```bash
synapse hub --db ~/synapse/hub.db
```

Then the client is a few `async` calls. The guarantee from section 3 — a claim
refuses an overlap before two agents edit the same file — is the `claim` method.
Run the cell below once a hub is listening on `ws://localhost:8876`.

In [ ]:
import asyncio

from synapse_channel import SynapseAgent


async def first_coordination() -> None:
    """Claim a file scope, mark the task working, then release it."""
    agent = SynapseAgent("ALPHA", uri="ws://localhost:8876")
    session = asyncio.create_task(agent.connect())
    if not await agent.wait_until_ready():
        raise RuntimeError("could not reach the hub — is `synapse hub` running?")

    await agent.claim("refactor-parser", note="splitting the tokenizer", paths=["src/parser"])
    await agent.update_task("refactor-parser", status="working")
    await agent.release("refactor-parser")

    agent.running = False
    session.cancel()


# In a notebook the event loop is already running, so await directly:
await first_coordination()

## Where to go next

- [Why SYNAPSE CHANNEL](https://github.com/anulum/synapse-channel/blob/main/docs/why-synapse.md) — what it is and why it matters.
- [Quick start](https://github.com/anulum/synapse-channel/blob/main/docs/quickstart.md) — the full golden path and coding-agent wiring.
- [Python API reference](https://github.com/anulum/synapse-channel/blob/main/docs/api.md) — the full client surface.